# 1s and 0s

<!-- - Sequence: Pediction Sentence
- Labels: Prediction Properties

Tasks
1. Update the prompt from zero-shot to few-show
2. How can we measure the output/labels from the LLMs?
    1. Could use multiple LLMs and perform majority vote for the labels
3. Perform on various datasets
    1. Synthetic data (which is below)
    2. Real datasets
        1.  financial_phrasebank
        2. chronicle2050 -->

> From `script_experiments/extract_prediction_properties.py`, where we take in text-based predictions and extract their properties---source, target, date, and outcome.

In [1]:
import os
import sys
import pprint

import pandas as pd

from tqdm import tqdm

notebook_dir = os.getcwd()

sys.path.append(os.path.join(notebook_dir, '../'))

from data_processing import DataProcessing
from prediction_properties import PredictionProperties
from text_generation_models import TextGenerationModelFactory
from vector_stores import ChromaVectorStore, VectorStoreDirector

In [2]:
# data/extract_prediction_properties/batch_2-from_df/llama-3.1-8b-instant/results.csv

In [3]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
extraction_path = "extract_prediction_properties/batch_1-from_df/llama-3.1-8b-instant"
dataset = os.path.join(base_data_path, extraction_path, 'results.csv')
df = DataProcessing.load_from_file(dataset, sep=',')
df

,Sentence,Raw Response,Model Name,No Property,Source,Target,Date,Outcome,Dataset Source
0,JPMorgan Chase forecasts that the net profit a...,"{\n0: [""JPMorgan"", ""Chase"", ""forecasts"", ""that...",llama-3.1-8b-instant,"JPMorgan, Chase, forecasts, that, the, net, pr...","JPMorgan, Chase",Amazon,"Q3, of, 2027","potentially, decrease",batch_1-from_df
1,"On August 21, 2024, Bank of America speculates...","{\n0: [""On"", ""August"", ""21"", ""2024"", ""will"", ""...",llama-3.1-8b-instant,"On, August, 21, 2024, will, likely, increase","Bank, of, America","the, revenue, at, Microsoft","August, 21, 2024",increase,batch_1-from_df
2,"Citigroup predicts on 2024-08-21, the operatin...","{\n 0: [""Citigroup"", ""predicts""],\n 1: [...",llama-3.1-8b-instant,"Citigroup, predicts",Citigroup,"the, operating, income, at, Alphabet",2024-08-21,"the, operating, income, may, rise",batch_1-from_df
3,"According to Goldman Sachs, the research and d...","{0: [""According"", ""to""], 1: [""Goldman"", ""Sachs...",llama-3.1-8b-instant,"According, to","Goldman, Sachs","the, research, and, development, expenses, at,...","would, fall, in, 2025","in, 2025",batch_1-from_df
4,"In 21 August 2024, Morgan Stanley envisions th...","{\n ""0"": [""In"", ""the"", ""gross"", ""profit"", ""at...",llama-3.1-8b-instant,"In, the, gross, profit, at, Johnson, &, Johnso...","Morgan, Stanley","the, gross, profit, at, Johnson, &, Johnson","21, August, 2024","remain, stable",batch_1-from_df
...,...,...,...,...,...,...,...,...,...
67,"On August 15, 2024, marketing expert David Lee...","{\n0: [""On"", ""August"", ""15,"", ""2024,""],\n1: [""...",llama-3.1-8b-instant,"On, August, 15,, 2024,","David, Lee","the, customer, satisfaction, ratings, at, Amazon","August, 15,, 2024","will, likely, increase",batch_1-from_df
68,"Professor James Davis predicts on November 20,...","{\n0: [""Professor"", ""James"", ""Davis"", ""predict...",llama-3.1-8b-instant,"Professor, James, Davis, predicts","Professor, James, Davis","the, average, salary, at, Google, may, rise","November, 20, 2025","the, average, salary, may, rise",batch_1-from_df
69,"According to economist Emily Patel, the unempl...","{\n 0: [""According"", ""to"", ""economist"", ""Em...",llama-3.1-8b-instant,"According, to, economist, Emily, Patel, the, u...","economist, Emily, Patel","unemployment, rate, at, the, United, States","January, 2029",fall,batch_1-from_df
70,"In 2024-08-20, research advisor Michael Brown ...","{0: [""In"", ""2024-08-20"", ""research"", ""advisor""...",llama-3.1-8b-instant,"In, 2024-08-20, research, advisor, Michael, Br...","Michael, Brown","the, success, rate, of, startups, in, Silicon,...",2024-08-20,"remain, stable",batch_1-from_df


In [8]:
properties_df = df.loc[: , ['Source', 'Target', 'Date', 'Outcome']]
converted_properties_df = properties_df.notna().astype(int)
converted_properties_df.rename(columns={'Source': 'Source (int)', 'Target': 'Target (int)', 'Date': 'Date (int)', 'Outcome': 'Outcome (int)'}, inplace=True)
converted_properties_df['Total Properties'] = converted_properties_df.sum(axis=1)

In [9]:
result_df = DataProcessing.concat_dfs([df, converted_properties_df], axis=1, ignore_index=False)
result_df

,Sentence,Raw Response,Model Name,No Property,Source,Target,Date,Outcome,Dataset Source,Source (int),Target (int),Date (int),Outcome (int),Total Properties
0,JPMorgan Chase forecasts that the net profit a...,"{\n0: [""JPMorgan"", ""Chase"", ""forecasts"", ""that...",llama-3.1-8b-instant,"JPMorgan, Chase, forecasts, that, the, net, pr...","JPMorgan, Chase",Amazon,"Q3, of, 2027","potentially, decrease",batch_1-from_df,1,1,1,1,4
1,"On August 21, 2024, Bank of America speculates...","{\n0: [""On"", ""August"", ""21"", ""2024"", ""will"", ""...",llama-3.1-8b-instant,"On, August, 21, 2024, will, likely, increase","Bank, of, America","the, revenue, at, Microsoft","August, 21, 2024",increase,batch_1-from_df,1,1,1,1,4
2,"Citigroup predicts on 2024-08-21, the operatin...","{\n 0: [""Citigroup"", ""predicts""],\n 1: [...",llama-3.1-8b-instant,"Citigroup, predicts",Citigroup,"the, operating, income, at, Alphabet",2024-08-21,"the, operating, income, may, rise",batch_1-from_df,1,1,1,1,4
3,"According to Goldman Sachs, the research and d...","{0: [""According"", ""to""], 1: [""Goldman"", ""Sachs...",llama-3.1-8b-instant,"According, to","Goldman, Sachs","the, research, and, development, expenses, at,...","would, fall, in, 2025","in, 2025",batch_1-from_df,1,1,1,1,4
4,"In 21 August 2024, Morgan Stanley envisions th...","{\n ""0"": [""In"", ""the"", ""gross"", ""profit"", ""at...",llama-3.1-8b-instant,"In, the, gross, profit, at, Johnson, &, Johnso...","Morgan, Stanley","the, gross, profit, at, Johnson, &, Johnson","21, August, 2024","remain, stable",batch_1-from_df,1,1,1,1,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,"On August 15, 2024, marketing expert David Lee...","{\n0: [""On"", ""August"", ""15,"", ""2024,""],\n1: [""...",llama-3.1-8b-instant,"On, August, 15,, 2024,","David, Lee","the, customer, satisfaction, ratings, at, Amazon","August, 15,, 2024","will, likely, increase",batch_1-from_df,1,1,1,1,4
68,"Professor James Davis predicts on November 20,...","{\n0: [""Professor"", ""James"", ""Davis"", ""predict...",llama-3.1-8b-instant,"Professor, James, Davis, predicts","Professor, James, Davis","the, average, salary, at, Google, may, rise","November, 20, 2025","the, average, salary, may, rise",batch_1-from_df,1,1,1,1,4
69,"According to economist Emily Patel, the unempl...","{\n 0: [""According"", ""to"", ""economist"", ""Em...",llama-3.1-8b-instant,"According, to, economist, Emily, Patel, the, u...","economist, Emily, Patel","unemployment, rate, at, the, United, States","January, 2029",fall,batch_1-from_df,1,1,1,1,4
70,"In 2024-08-20, research advisor Michael Brown ...","{0: [""In"", ""2024-08-20"", ""research"", ""advisor""...",llama-3.1-8b-instant,"In, 2024-08-20, research, advisor, Michael, Br...","Michael, Brown","the, success, rate, of, startups, in, Silicon,...",2024-08-20,"remain, stable",batch_1-from_df,1,1,1,1,4
